# Powerlifting Silver Layer

Cleans and deduplicates the bronze snapshots into `workspace.powerlifting_silver`: typed columns, decoded enums (SexLabel, FinishStatus), failed-attempt negatives set to NULL, and latest-batch dedup per natural key.

In [0]:
%sql
-- Silver layer: cleaned, deduplicated, typed tables.
-- Bronze was appended (full Kaggle re-load each run), so dedupe keeps the latest
-- ingested batch per natural key and drops load-provenance columns.

CREATE SCHEMA IF NOT EXISTS workspace.powerlifting_silver;

-- meets: natural key = MeetID (unique within a snapshot). Keep latest batch.
CREATE OR REPLACE TABLE workspace.powerlifting_silver.meets AS
WITH latest AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY MeetID ORDER BY _ingested_at DESC) AS _rn
  FROM workspace.powerlifting_bronze.meets
)
SELECT
  MeetID,
  trim(MeetPath)    AS MeetPath,
  trim(Federation)  AS Federation,
  to_date(Date, 'yyyy-MM-dd') AS MeetDate,
  trim(MeetCountry) AS MeetCountry,
  trim(MeetState)   AS MeetState,
  trim(MeetTown)    AS MeetTown,
  trim(MeetName)    AS MeetName
FROM latest
WHERE _rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- openpowerlifting: natural key = (MeetID, Name, Sex, Equipment, Division, WeightClassKg).
-- A lifter can appear multiple times under the same key with partial results
-- (e.g. bench-only vs full-power) or exact source duplicates. Dedupe keeps the
-- latest ingested batch, then the row with the highest TotalKg (most complete result).
CREATE OR REPLACE TABLE workspace.powerlifting_silver.openpowerlifting AS
WITH latest AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY MeetID, Name, Sex, Equipment, Division, WeightClassKg
      ORDER BY _ingested_at DESC, TotalKg DESC NULLS LAST, BestSquatKg DESC NULLS LAST
    ) AS _rn
  FROM workspace.powerlifting_bronze.openpowerlifting
)
SELECT
  MeetID,
  trim(Name)        AS Name,
  Sex,
  CASE Sex WHEN 'M' THEN 'Male' WHEN 'F' THEN 'Female' ELSE NULL END AS SexLabel,
  trim(Equipment)   AS Equipment,
  CAST(Age AS DECIMAL(5,1)) AS Age,
  trim(Division)    AS Division,
  BodyweightKg,
  trim(WeightClassKg) AS WeightClassKg,
  -- 4th attempts: negative = failed attempt -> NULL
  CASE WHEN Squat4Kg    >= 0 THEN Squat4Kg    END AS Squat4Kg,
  CASE WHEN Bench4Kg     >= 0 THEN Bench4Kg     END AS Bench4Kg,
  CASE WHEN Deadlift4Kg  >= 0 THEN Deadlift4Kg  END AS Deadlift4Kg,
  -- Best lifts: negative = no successful lift -> NULL
  CASE WHEN BestSquatKg    >= 0 THEN BestSquatKg    END AS BestSquatKg,
  CASE WHEN BestBenchKg     >= 0 THEN BestBenchKg     END AS BestBenchKg,
  CASE WHEN BestDeadliftKg  >= 0 THEN BestDeadliftKg  END AS BestDeadliftKg,
  TotalKg,
  -- Place: numeric placement -> INT, non-numeric codes -> NULL
  CASE WHEN Place RLIKE '^[0-9]+$' THEN CAST(Place AS INT) END AS Place,
  -- FinishStatus enum decoded from Place codes
  CASE
    WHEN Place RLIKE '^[0-9]+$' THEN 'Finished'
    WHEN Place = 'DQ' THEN 'Disqualified'
    WHEN Place = 'G'  THEN 'Guest'
    WHEN Place = 'DD' THEN 'Did Not Weigh In'
    WHEN Place = 'NS' THEN 'No Show'
    ELSE NULL
  END AS FinishStatus,
  Wilks
FROM latest
WHERE _rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Verify silver layer
SELECT 'meets' AS table_name, COUNT(*) AS row_count FROM workspace.powerlifting_silver.meets
UNION ALL
SELECT 'openpowerlifting', COUNT(*) FROM workspace.powerlifting_silver.openpowerlifting;

-- FinishStatus distribution (should account for all non-null Place)
SELECT FinishStatus, COUNT(*) AS cnt
FROM workspace.powerlifting_silver.openpowerlifting
GROUP BY 1
ORDER BY 2 DESC;

-- Confirm no negative values survived in lift columns
SELECT
  SUM(CASE WHEN BestSquatKg    < 0 THEN 1 ELSE 0 END) AS neg_squat,
  SUM(CASE WHEN BestBenchKg     < 0 THEN 1 ELSE 0 END) AS neg_bench,
  SUM(CASE WHEN BestDeadliftKg  < 0 THEN 1 ELSE 0 END) AS neg_dead,
  SUM(CASE WHEN Squat4Kg    < 0 THEN 1 ELSE 0 END) AS neg_squat4,
  SUM(CASE WHEN Place IS NOT NULL AND FinishStatus IS NULL THEN 1 ELSE 0 END) AS place_without_status,
  SUM(CASE WHEN Place IS NULL AND FinishStatus IS NOT NULL THEN 1 ELSE 0 END) AS status_without_place
FROM workspace.powerlifting_silver.openpowerlifting;

-- Sample cleaned rows
SELECT * FROM workspace.powerlifting_silver.openpowerlifting LIMIT 5;

MeetID,Name,Sex,SexLabel,Equipment,Age,Division,BodyweightKg,WeightClassKg,Squat4Kg,Bench4Kg,Deadlift4Kg,BestSquatKg,BestBenchKg,BestDeadliftKg,TotalKg,Place,FinishStatus,Wilks
0,Al Reiss,M,Male,Wraps,61.0,Mst 60-64,101.79,110,176.9,null,null,170.1,104.33,167.83,442.26,1,Finished,267.3
0,Alexis Eliopoulos,F,Female,Wraps,30.0,Senior,88.36,90,null,null,204.12,174.63,122.47,195.04,492.14,1,Finished,428.59
0,Andrew Mooe,M,Male,Wraps,27.0,Open Senior,99.25,100,null,null,null,226.8,163.29,272.16,662.25,2,Finished,404.26
0,Angie Belk Terry,F,Female,Wraps,47.0,Mst 45-49,59.6,60,null,null,null,47.63,20.41,70.31,138.35,1,Finished,155.05
0,Barron Henderson,M,Male,Single-ply,40.0,Mst 40-44,118.66,125,null,null,null,362.87,181.44,301.64,845.95,1,Finished,487.65
